# NutriLens — Colab Training Notebook

Run cells top-to-bottom. Runtime → Change runtime type → **GPU: A100** before starting.

Expected wall-clock on A100:
- ResNet-50, 15 epochs: ~45 min
- ViT-B/16, 20 epochs: ~1.5 hr
- Frozen-backbone ablations (head-only): ~15 min each

## 1. Verify GPU

In [ ]:
!nvidia-smi -L
import torch
print('CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Mount Google Drive (for persistent checkpoints + .env)

Before running this, put your project folder in Drive at `MyDrive/NutriLens/` and copy your local `.env` there too.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/NutriLens/

## 3. Sync project to Colab's fast SSD

We train from `/content/project/` (fast SSD). Checkpoints land in Drive for persistence.

In [ ]:
!rm -rf /content/project && cp -r /content/drive/MyDrive/NutriLens /content/project
!cp /content/drive/MyDrive/NutriLens/.env /content/project/.env
%cd /content/project
!ls

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5. Download Food-101 to fast SSD (~3 min on Colab network)

In [ ]:
!python scripts/download_data.py

## 6. Build USDA → FDC mapping (one-time, ~1 min)

Skip if `data/food101_fdc_map.json` already exists.

In [ ]:
import os
if not os.path.exists('data/food101_fdc_map.json'):
    !python scripts/build_mapping.py
    !cp data/food101_fdc_map.json /content/drive/MyDrive/NutriLens/data/
else:
    print('mapping already exists')

## 7. Training run: ResNet-50 fine-tune (~45 min)

In [ ]:
!python -m classifier.train \
  --arch resnet50 \
  --epochs 15 \
  --batch_size 128 \
  --lr 3e-4 \
  --num_workers 4

## 8. Training run: ViT-B/16 fine-tune (~1.5 hr)

In [ ]:
!python -m classifier.train \
  --arch vit_b_16 \
  --epochs 20 \
  --batch_size 64 \
  --lr 1e-4 \
  --num_workers 4

## 9. Ablation runs (for rubric item #91: ablation study)

Short head-only runs. Frozen backbone + augmentation on/off.

In [ ]:
!python -m classifier.train --arch resnet50 --epochs 8 --frozen --batch_size 128
!python -m classifier.train --arch resnet50 --epochs 8 --frozen --no_augment --batch_size 128
!python -m classifier.train --arch vit_b_16 --epochs 8 --frozen --batch_size 64

## 10. Persist checkpoints back to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/NutriLens/checkpoints
!cp -v checkpoints/* /content/drive/MyDrive/NutriLens/checkpoints/

## 11. Run full eval on Food-101 test split

In [ ]:
!python -m eval.run --checkpoint checkpoints/vit_b_16_best.pt
!python -m eval.run --checkpoint checkpoints/resnet50_best.pt
!python -m eval.ablation
!cp -rv eval/outputs /content/drive/MyDrive/NutriLens/eval/